# Inteligencia de Mercado — Protege Bank

**Voz do Cliente vs Comportamento Real · 50.000 Clientes · NPS · Churn · Market Fit**

[![Python](https://img.shields.io/badge/Python-3.11-blue?logo=python)](https://python.org)
[![Jupyter](https://img.shields.io/badge/Jupyter-Notebook-orange?logo=jupyter)](https://jupyter.org)

---

> *"O cliente disse que estava satisfeito. O modelo sabia que ele ia embora em 60 dias. Qual voce acredita?"*

Este case aplica **Inteligencia de Mercado** ao ecossistema bancario, triangulando a **voz declarada do cliente**
(NPS, satisfacao, pesquisa) com o **comportamento real** (transacoes, engajamento digital, churn)
para gerar insights acionaveis sobre Market Fit, retencao e posicionamento de produto.

**Camadas de analise:**
- Pesquisa Quantitativa: NPS e satisfacao por segmento
- Benchmarking Competitivo: participacao de mercado e penetracao de produto
- Triangulacao: NPS declarado vs comportamento real
- Market Fit: avaliacao por produto e perfil demografico
- Churn Prediction: modelo preditivo com Random Forest + XGBoost
- Visao 360: dashboard executivo integrando todas as camadas

## Secao 0 — Imports e Configuracoes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, roc_curve
import warnings, os

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.family']   = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')

os.makedirs('images', exist_ok=True)

# Paleta Protege Bank
PROTEGE_BLUE  = '#003087'
PROTEGE_RED   = '#E31837'
PROTEGE_GOLD  = '#FFB800'
PROTEGE_GREEN = '#00A651'
PROTEGE_GRAY  = '#6C757D'

CORES_NPS = {'Promotor': PROTEGE_GREEN, 'Neutro': PROTEGE_GOLD, 'Detrator': PROTEGE_RED}
CORES_SEG = {
    'Bronze': '#CD7F32', 'Prata': '#C0C0C0',
    'Ouro': PROTEGE_GOLD, 'Diamante': '#00BFFF'
}

N = 50_000
print('Ambiente configurado. Protege Bank — Inteligencia de Mercado')

## Secao 1 — Geracao da Base de Clientes

Base sintetica de **50.000 clientes** do Protege Bank com:
- Perfil demografico (idade, renda, regiao, profissao)
- Portfolio de produtos (cartao, seguro, financiamento, investimento)
- Dados comportamentais (transacoes, engajamento digital, reclamacoes)
- Pesquisa declarada (NPS, satisfacao por produto)
- Indicador de churn (realidade vs declaracao)

In [ ]:
regioes    = ['Sudeste','Sul','Nordeste','Centro-Oeste','Norte']
pesos_reg  = [0.45, 0.18, 0.20, 0.10, 0.07]
profissoes = ['CLT','Autonomo','Empresario','Servidor Publico','Aposentado','Estudante']
produtos   = ['Cartao de Credito','Seguro Auto','Seguro Vida','Financiamento','Investimento','Consorcio']

df = pd.DataFrame()
df['cliente_id']  = [f'PB{str(i).zfill(6)}' for i in range(1, N+1)]
df['idade']       = np.random.normal(38, 12, N).clip(18, 75).astype(int)
df['renda_mensal'] = np.random.lognormal(8.5, 0.6, N).clip(1500, 80000).round(0)
df['regiao']      = np.random.choice(regioes, N, p=pesos_reg)
df['profissao']   = np.random.choice(profissoes, N, p=[0.40,0.20,0.12,0.10,0.10,0.08])
df['tempo_cliente_anos'] = np.random.exponential(4, N).clip(0.1, 25).round(1)
df['n_produtos']  = np.random.choice([1,2,3,4,5,6], N, p=[0.30,0.28,0.20,0.12,0.07,0.03])

# Segmento baseado em renda e produtos
def segmentar(row):
    if row['renda_mensal'] >= 15000 and row['n_produtos'] >= 4: return 'Diamante'
    elif row['renda_mensal'] >= 7000 and row['n_produtos'] >= 3: return 'Ouro'
    elif row['renda_mensal'] >= 3000 and row['n_produtos'] >= 2: return 'Prata'
    else: return 'Bronze'
df['segmento'] = df.apply(segmentar, axis=1)

# Comportamento digital
df['acessos_app_mes']    = np.random.poisson(12, N).clip(0, 60)
df['transacoes_mes']     = np.random.poisson(8, N).clip(0, 50)
df['reclamacoes_12m']    = np.random.choice([0,1,2,3,4], N, p=[0.65,0.20,0.10,0.03,0.02])
df['dias_ultimo_acesso'] = np.random.exponential(15, N).clip(1, 180).astype(int)

# NPS declarado (1-10)
def gerar_nps(row):
    base = 7.0
    if row['segmento'] == 'Diamante': base += 1.5
    elif row['segmento'] == 'Ouro': base += 0.8
    elif row['segmento'] == 'Bronze': base -= 0.5
    if row['reclamacoes_12m'] > 1: base -= 1.5
    if row['tempo_cliente_anos'] > 5: base += 0.5
    return np.clip(np.random.normal(base, 1.5), 1, 10).round(0).astype(int)
df['nps_score'] = df.apply(gerar_nps, axis=1)

def classificar_nps(score):
    if score >= 9: return 'Promotor'
    elif score >= 7: return 'Neutro'
    else: return 'Detrator'
df['nps_categoria'] = df['nps_score'].apply(classificar_nps)

# Satisfacao por produto (1-5)
for prod in ['cartao','seguro','app_digital','atendimento','taxas']:
    base = np.random.normal(3.8, 0.8, N).clip(1, 5)
    if prod == 'taxas': base -= 0.4
    if prod == 'app_digital': base += 0.2
    df[f'sat_{prod}'] = base.round(1)

# CHURN — comportamento real (nao declarado)
def calcular_prob_churn(row):
    prob = 0.08  # base
    if row['reclamacoes_12m'] >= 2: prob += 0.25
    if row['dias_ultimo_acesso'] > 60: prob += 0.20
    if row['acessos_app_mes'] < 3: prob += 0.15
    if row['n_produtos'] == 1: prob += 0.10
    if row['nps_score'] <= 6: prob += 0.12
    if row['tempo_cliente_anos'] < 1: prob += 0.10
    if row['segmento'] == 'Diamante': prob -= 0.10
    if row['segmento'] == 'Ouro': prob -= 0.05
    return min(prob, 0.95)

df['prob_churn']  = df.apply(calcular_prob_churn, axis=1)
df['churn']       = (np.random.random(N) < df['prob_churn']).astype(int)

# Paradoxo: clientes que dizem estar satisfeitos mas vao embora
df['paradoxo_nps'] = ((df['nps_categoria'] != 'Detrator') & (df['churn'] == 1)).astype(int)

print(f'Base gerada: {len(df):,} clientes')
print(f'Taxa de churn: {df["churn"].mean():.1%}')
print(f'Paradoxo NPS (satisfeitos que vao embora): {df["paradoxo_nps"].sum():,} ({df["paradoxo_nps"].mean():.1%})')
print()
print(df[['cliente_id','idade','renda_mensal','segmento','nps_categoria','churn']].head(8).to_string(index=False))

## Secao 2 — Perfil Demografico e Segmentacao

Analise da composicao da base de clientes por regiao, segmento, renda e perfil demografico.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Distribuicao por segmento
seg_counts = df['segmento'].value_counts()
ordem_seg  = ['Bronze','Prata','Ouro','Diamante']
seg_ord    = seg_counts.reindex(ordem_seg)
cores_seg_list = [CORES_SEG[s] for s in ordem_seg]
bars0 = axes[0,0].bar(ordem_seg, seg_ord, color=cores_seg_list, edgecolor='white')
axes[0,0].bar_label(bars0, labels=[f'{v:,}\n({v/N*100:.1f}%)' for v in seg_ord], padding=5, fontweight='bold')
axes[0,0].set_title('Distribuicao por Segmento', fontweight='bold', fontsize=13)
axes[0,0].set_ylabel('Clientes')
axes[0,0].set_ylim(0, seg_ord.max() * 1.25)

# Renda por segmento
dados_box = [df[df['segmento']==s]['renda_mensal'].values for s in ordem_seg]
bp = axes[0,1].boxplot(dados_box, labels=ordem_seg, patch_artist=True)
for patch, cor in zip(bp['boxes'], cores_seg_list):
    patch.set_facecolor(cor); patch.set_alpha(0.7)
axes[0,1].set_title('Renda Mensal por Segmento (R$)', fontweight='bold', fontsize=13)
axes[0,1].set_ylabel('Renda (R$)')
axes[0,1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,p: f'R${x/1e3:.0f}k'))

# Distribuicao regional
reg_counts = df['regiao'].value_counts()
bars2 = axes[1,0].bar(reg_counts.index, reg_counts.values, color=PROTEGE_BLUE, edgecolor='white', alpha=0.85)
axes[1,0].bar_label(bars2, labels=[f'{v:,}' for v in reg_counts.values], padding=5, fontweight='bold')
axes[1,0].set_title('Distribuicao Regional de Clientes', fontweight='bold', fontsize=13)
axes[1,0].tick_params(axis='x', rotation=20)

# Piramide etaria simplificada
bins = [18,25,35,45,55,65,76]
labels_idade = ['18-24','25-34','35-44','45-54','55-64','65+']
df['faixa_etaria'] = pd.cut(df['idade'], bins=bins, labels=labels_idade, right=False)
idade_counts = df['faixa_etaria'].value_counts().reindex(labels_idade)
axes[1,1].barh(labels_idade, idade_counts.values, color=PROTEGE_BLUE, edgecolor='white', alpha=0.85)
axes[1,1].bar_label(axes[1,1].containers[0], labels=[f'{v:,}' for v in idade_counts.values], padding=5, fontweight='bold')
axes[1,1].set_title('Distribuicao por Faixa Etaria', fontweight='bold', fontsize=13)

plt.suptitle('Perfil Demografico — Protege Bank | 50.000 Clientes', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('images/01_perfil_demografico.png', dpi=150, bbox_inches='tight')
plt.show()

## Secao 3 — Pesquisa Quantitativa: NPS e Satisfacao

Analise do **Net Promoter Score** e satisfacao declarada por produto, segmento e regiao.

O NPS e calculado como: `% Promotores - % Detratores`

In [ ]:
# NPS geral
prom = (df['nps_categoria'] == 'Promotor').mean() * 100
neut = (df['nps_categoria'] == 'Neutro').mean() * 100
detr = (df['nps_categoria'] == 'Detrator').mean() * 100
nps_geral = prom - detr

print(f'NPS GERAL: {nps_geral:.1f}')
print(f'  Promotores: {prom:.1f}%')
print(f'  Neutros:    {neut:.1f}%')
print(f'  Detratores: {detr:.1f}%')
print()

# NPS por segmento
nps_seg = df.groupby('segmento').apply(
    lambda x: (x['nps_categoria']=='Promotor').mean()*100 - (x['nps_categoria']=='Detrator').mean()*100
).reindex(ordem_seg)
print('NPS por segmento:')
for seg, nps in nps_seg.items():
    print(f'  {seg:<12}: {nps:.1f}')

# NPS por regiao
nps_reg = df.groupby('regiao').apply(
    lambda x: (x['nps_categoria']=='Promotor').mean()*100 - (x['nps_categoria']=='Detrator').mean()*100
)
print('\nNPS por regiao:')
for reg, nps in nps_reg.sort_values(ascending=False).items():
    print(f'  {reg:<20}: {nps:.1f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Pizza NPS geral
sizes = [prom, neut, detr]
labels_nps = [f'Promotores\n{prom:.1f}%', f'Neutros\n{neut:.1f}%', f'Detratores\n{detr:.1f}%']
cores_pizza = [PROTEGE_GREEN, PROTEGE_GOLD, PROTEGE_RED]
wedges, texts = axes[0,0].pie(sizes, labels=labels_nps, colors=cores_pizza,
    startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[0,0].set_title(f'NPS Geral: {nps_geral:.1f}', fontweight='bold', fontsize=14)

# NPS por segmento
cores_nps_bar = [PROTEGE_GREEN if v >= 0 else PROTEGE_RED for v in nps_seg]
bars1 = axes[0,1].bar(nps_seg.index, nps_seg.values, color=cores_nps_bar, edgecolor='white')
axes[0,1].bar_label(bars1, labels=[f'{v:.1f}' for v in nps_seg.values], padding=5, fontweight='bold')
axes[0,1].axhline(0, color='black', linewidth=0.8)
axes[0,1].set_title('NPS por Segmento', fontweight='bold', fontsize=13)
axes[0,1].set_ylabel('NPS')

# Satisfacao por produto (media)
prods = ['cartao','seguro','app_digital','atendimento','taxas']
labels_prod = ['Cartao','Seguro','App Digital','Atendimento','Taxas']
medias_prod = [df[f'sat_{p}'].mean() for p in prods]
cores_prod = [PROTEGE_GREEN if m >= 4 else PROTEGE_GOLD if m >= 3.5 else PROTEGE_RED for m in medias_prod]
bars2 = axes[1,0].barh(labels_prod, medias_prod, color=cores_prod, edgecolor='white')
axes[1,0].bar_label(bars2, labels=[f'{v:.2f}' for v in medias_prod], padding=5, fontweight='bold')
axes[1,0].axvline(3.5, color='red', linestyle='--', alpha=0.7, label='Minimo aceitavel: 3.5')
axes[1,0].set_xlim(0, 5.5)
axes[1,0].set_title('Satisfacao Media por Produto (1-5)', fontweight='bold', fontsize=13)
axes[1,0].legend()

# NPS por regiao
nps_reg_ord = nps_reg.sort_values(ascending=True)
cores_reg = [PROTEGE_GREEN if v >= 0 else PROTEGE_RED for v in nps_reg_ord]
bars3 = axes[1,1].barh(nps_reg_ord.index, nps_reg_ord.values, color=cores_reg, edgecolor='white')
axes[1,1].bar_label(bars3, labels=[f'{v:.1f}' for v in nps_reg_ord.values], padding=5, fontweight='bold')
axes[1,1].axvline(0, color='black', linewidth=0.8)
axes[1,1].set_title('NPS por Regiao', fontweight='bold', fontsize=13)

plt.suptitle('Pesquisa Quantitativa — NPS e Satisfacao | Protege Bank', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('images/02_nps_satisfacao.png', dpi=150, bbox_inches='tight')
plt.show()

## Secao 4 — Benchmarking Competitivo

Analise da penetracao de produtos e participacao de mercado no ecossistema bancario brasileiro,
comparando o Protege Bank com os principais concorrentes.

In [ ]:
# Penetracao de produtos na base Protege Bank
penetracao_porto = {
    'Cartao de Credito': 0.72,
    'Seguro Auto':       0.45,
    'Seguro Vida':       0.28,
    'Financiamento':     0.31,
    'Investimento':      0.22,
    'Consorcio':         0.15,
}

# Benchmarking de mercado (dados publicos estimados)
benchmark = pd.DataFrame({
    'Produto':        list(penetracao_porto.keys()),
    'Protege Bank':     list(penetracao_porto.values()),
    'Itau':           [0.85, 0.52, 0.38, 0.45, 0.41, 0.20],
    'Bradesco':       [0.80, 0.48, 0.42, 0.40, 0.35, 0.18],
    'Nubank':         [0.95, 0.15, 0.08, 0.12, 0.30, 0.05],
    'Media Mercado':  [0.75, 0.38, 0.28, 0.32, 0.30, 0.14],
})

print('Benchmarking Competitivo — Penetracao de Produtos:')
print(benchmark.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Grouped bar — penetracao por produto
x = np.arange(len(benchmark))
w = 0.18
players = ['Protege Bank','Itau','Bradesco','Nubank','Media Mercado']
cores_bench = [PROTEGE_BLUE, '#FF6B00', '#CC0000', '#820AD1', PROTEGE_GRAY]

for i, (player, cor) in enumerate(zip(players, cores_bench)):
    lw = 2.5 if player == 'Protege Bank' else 1.5
    bars = axes[0].bar(x + i*w, benchmark[player]*100, w,
                       label=player, color=cor, edgecolor='white', linewidth=lw,
                       alpha=0.9 if player == 'Protege Bank' else 0.7)

axes[0].set_xticks(x + w*2)
axes[0].set_xticklabels(benchmark['Produto'], rotation=20, ha='right')
axes[0].set_title('Penetracao de Produtos por Banco (%)', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Penetracao (%)')
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 110)

# GAP Protege Bank vs media mercado
benchmark['gap'] = benchmark['Protege Bank'] - benchmark['Media Mercado']
cores_gap = [PROTEGE_GREEN if v >= 0 else PROTEGE_RED for v in benchmark['gap']]
bars2 = axes[1].barh(benchmark['Produto'], benchmark['gap']*100, color=cores_gap, edgecolor='white')
axes[1].bar_label(bars2, labels=[f'{v*100:+.1f}pp' for v in benchmark['gap']], padding=5, fontweight='bold')
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_title('GAP Protege Bank vs Media de Mercado (pp)', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Pontos percentuais')

plt.suptitle('Benchmarking Competitivo — Ecossistema Bancario Brasileiro', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('images/03_benchmarking.png', dpi=150, bbox_inches='tight')
plt.show()

## Secao 5 — Triangulacao: Voz Declarada vs Comportamento Real

**O insight mais poderoso de Inteligencia de Mercado:**
cruzar o que o cliente diz com o que ele realmente faz.

| Grupo | NPS Declarado | Comportamento Real |
|-------|--------------|-------------------|
| Alinhado Positivo | Promotor | Fica |
| Alinhado Negativo | Detrator | Vai embora |
| **Paradoxo Silencioso** | **Neutro/Promotor** | **Vai embora** |
| Surpresa Positiva | Detrator | Fica |

In [ ]:
# Triangulacao NPS vs Churn
def classificar_triangulacao(row):
    if row['nps_categoria'] == 'Promotor' and row['churn'] == 0:   return 'Alinhado Positivo'
    elif row['nps_categoria'] == 'Detrator' and row['churn'] == 1: return 'Alinhado Negativo'
    elif row['nps_categoria'] != 'Detrator' and row['churn'] == 1: return 'Paradoxo Silencioso'
    else: return 'Surpresa Positiva'

df['triangulacao'] = df.apply(classificar_triangulacao, axis=1)

print('TRIANGULACAO: VOZ DECLARADA vs COMPORTAMENTO REAL')
print('=' * 55)
for grupo, n in df['triangulacao'].value_counts().items():
    pct = n/len(df)*100
    print(f'  {grupo:<25} {n:>7,} clientes ({pct:.1f}%)')

paradoxo = df[df['triangulacao'] == 'Paradoxo Silencioso']
print(f'\nParadoxo Silencioso:')
print(f'  Clientes que NAO sao Detratores mas vao embora: {len(paradoxo):,}')
print(f'  NPS medio desse grupo: {paradoxo["nps_score"].mean():.1f}')
print(f'  Acessos app/mes (media): {paradoxo["acessos_app_mes"].mean():.1f}')
print(f'  Reclamacoes 12m (media): {paradoxo["reclamacoes_12m"].mean():.2f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# Distribuicao da triangulacao
tri_counts = df['triangulacao'].value_counts()
cores_tri  = {  'Alinhado Positivo': PROTEGE_GREEN,
                'Paradoxo Silencioso': PROTEGE_RED,
                'Surpresa Positiva': PROTEGE_GOLD,
                'Alinhado Negativo': PROTEGE_GRAY }
wedges, texts, autotexts = axes[0].pie(
    tri_counts.values,
    labels=tri_counts.index,
    colors=[cores_tri.get(k, PROTEGE_GRAY) for k in tri_counts.index],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor':'white','linewidth':2})
for at in autotexts: at.set_fontweight('bold')
axes[0].set_title('Triangulacao: NPS vs Churn Real', fontweight='bold', fontsize=12)

# Heatmap NPS categoria vs Churn
cross = pd.crosstab(df['nps_categoria'], df['churn'], normalize='index') * 100
cross.columns = ['Ficou','Churnou']
cross = cross.reindex(['Promotor','Neutro','Detrator'])
sns.heatmap(cross, annot=True, fmt='.1f', cmap='RdYlGn_r',
            ax=axes[1], cbar_kws={'label':'%'},
            linewidths=1, annot_kws={'fontsize':13,'fontweight':'bold'})
axes[1].set_title('Taxa de Churn por Categoria NPS (%)', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Resultado Real')
axes[1].set_ylabel('NPS Declarado')

# Comportamento do Paradoxo Silencioso
metricas_paradoxo = {
    'Acessos App/mes':     [df[df['churn']==0]['acessos_app_mes'].mean(),
                            paradoxo['acessos_app_mes'].mean()],
    'Transacoes/mes':      [df[df['churn']==0]['transacoes_mes'].mean(),
                            paradoxo['transacoes_mes'].mean()],
    'Dias sem acesso':     [df[df['churn']==0]['dias_ultimo_acesso'].mean(),
                            paradoxo['dias_ultimo_acesso'].mean()],
    'Reclamacoes 12m':     [df[df['churn']==0]['reclamacoes_12m'].mean(),
                            paradoxo['reclamacoes_12m'].mean()],
}
x = np.arange(len(metricas_paradoxo))
w = 0.35
vals_base = [v[0] for v in metricas_paradoxo.values()]
vals_par  = [v[1] for v in metricas_paradoxo.values()]
axes[2].bar(x - w/2, vals_base, w, label='Clientes Retidos', color=PROTEGE_GREEN, edgecolor='white', alpha=0.85)
axes[2].bar(x + w/2, vals_par,  w, label='Paradoxo Silencioso', color=PROTEGE_RED, edgecolor='white', alpha=0.85)
axes[2].set_xticks(x)
axes[2].set_xticklabels(list(metricas_paradoxo.keys()), rotation=20, ha='right', fontsize=9)
axes[2].set_title('Comportamento: Retidos vs Paradoxo Silencioso', fontweight='bold', fontsize=12)
axes[2].legend(fontsize=9)

plt.suptitle('Triangulacao de Dados — Voz Declarada vs Comportamento Real', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('images/04_triangulacao.png', dpi=150, bbox_inches='tight')
plt.show()

## Secao 6 — Market Fit por Produto e Segmento

Avaliacao do **ajuste de mercado** de cada produto Protege Bank,
cruzando penetracao, satisfacao e retencao por perfil de cliente.

In [ ]:
# Score de Market Fit por segmento e produto
produtos_sat = ['cartao','seguro','app_digital','atendimento','taxas']
labels_mf    = ['Cartao','Seguro','App Digital','Atendimento','Taxas']

market_fit = []
for seg in ordem_seg:
    df_seg = df[df['segmento'] == seg]
    for prod, label in zip(produtos_sat, labels_mf):
        sat_media  = df_seg[f'sat_{prod}'].mean()
        nps_medio  = df_seg['nps_score'].mean()
        retencao   = 1 - df_seg['churn'].mean()
        score_mf   = (sat_media/5 * 0.4 + nps_medio/10 * 0.3 + retencao * 0.3) * 100
        market_fit.append({'segmento': seg, 'produto': label,
                           'satisfacao': sat_media, 'nps_medio': nps_medio,
                           'retencao': retencao * 100, 'score_mf': score_mf})

df_mf = pd.DataFrame(market_fit)

# Pivot para heatmap
pivot_mf = df_mf.pivot(index='segmento', columns='produto', values='score_mf')
pivot_mf = pivot_mf.reindex(ordem_seg)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Heatmap Market Fit
sns.heatmap(pivot_mf, annot=True, fmt='.1f', cmap='RdYlGn',
            ax=axes[0], vmin=50, vmax=90,
            linewidths=1, annot_kws={'fontsize':12,'fontweight':'bold'},
            cbar_kws={'label':'Score de Market Fit (0-100)'})
axes[0].set_title('Score de Market Fit por Segmento e Produto', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Produto')
axes[0].set_ylabel('Segmento')

# Score medio por segmento
mf_seg = df_mf.groupby('segmento')['score_mf'].mean().reindex(ordem_seg)
cores_mf = [CORES_SEG[s] for s in ordem_seg]
bars = axes[1].bar(ordem_seg, mf_seg, color=cores_mf, edgecolor='white')
axes[1].bar_label(bars, labels=[f'{v:.1f}' for v in mf_seg], padding=5, fontweight='bold', fontsize=12)
axes[1].axhline(70, color='red', linestyle='--', alpha=0.7, label='Meta minima: 70')
axes[1].set_title('Score Medio de Market Fit por Segmento', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Score de Market Fit')
axes[1].set_ylim(0, 100)
axes[1].legend()

plt.suptitle('Avaliacao de Market Fit — Protege Bank', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('images/05_market_fit.png', dpi=150, bbox_inches='tight')
plt.show()

## Secao 7 — Churn Prediction: Modelo Preditivo

Modelo de **classificacao** para identificar clientes com alto risco de churn
antes que o comportamento de saida se torne irreversivel.

**Features utilizadas:** comportamento digital, reclamacoes, produtos, segmento, NPS, tempo de relacionamento

In [ ]:
# Preparar features
features = ['idade','renda_mensal','tempo_cliente_anos','n_produtos',
            'acessos_app_mes','transacoes_mes','reclamacoes_12m',
            'dias_ultimo_acesso','nps_score',
            'sat_cartao','sat_seguro','sat_app_digital','sat_atendimento','sat_taxas',
            'segmento','regiao']

df_model = df[features + ['churn']].copy()

# Encoding categoricas
les = {}
for col in ['segmento','regiao']:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    les[col] = le

X = df_model.drop('churn', axis=1)
y = df_model['churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred  = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_proba)
cv  = cross_val_score(rf, X, y, cv=5, scoring='roc_auc').mean()

print('RANDOM FOREST — CHURN PREDICTION')
print('=' * 45)
print(f'  AUC-ROC:       {auc:.4f}')
print(f'  AUC-ROC CV:    {cv:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Ficou','Churnou']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[0].plot(fpr, tpr, color=PROTEGE_BLUE, linewidth=2.5, label=f'AUC = {auc:.3f}')
axes[0].plot([0,1],[0,1], 'k--', linewidth=1, alpha=0.5, label='Aleatorio')
axes[0].fill_between(fpr, tpr, alpha=0.1, color=PROTEGE_BLUE)
axes[0].set_title('Curva ROC — Modelo de Churn', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Taxa de Falso Positivo')
axes[0].set_ylabel('Taxa de Verdadeiro Positivo')
axes[0].legend(fontsize=11)

# Feature Importance
importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True).tail(12)
bars = axes[1].barh(importancias.index, importancias.values * 100, color=PROTEGE_BLUE, edgecolor='white', alpha=0.85)
axes[1].bar_label(bars, labels=[f'{v*100:.1f}%' for v in importancias.values], padding=3, fontsize=9)
axes[1].set_title('Variaveis mais Importantes para o Churn', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Importancia (%)')

# Distribuicao de probabilidade de churn por segmento
df_test = X_test.copy()
df_test['prob_churn_pred'] = y_proba
df_test['segmento_nome']   = les['segmento'].inverse_transform(df_test['segmento'])
dados_violin = [df_test[df_test['segmento_nome']==s]['prob_churn_pred'].values for s in ordem_seg]
vp = axes[2].violinplot(dados_violin, positions=range(len(ordem_seg)), showmedians=True)
for body, cor in zip(vp['bodies'], cores_mf):
    body.set_facecolor(cor); body.set_alpha(0.7)
axes[2].set_xticks(range(len(ordem_seg)))
axes[2].set_xticklabels(ordem_seg)
axes[2].set_title('Probabilidade de Churn por Segmento', fontweight='bold', fontsize=13)
axes[2].set_ylabel('Probabilidade de Churn')
axes[2].axhline(0.5, color='red', linestyle='--', alpha=0.7, label='Limiar de risco: 50%')
axes[2].legend()

plt.suptitle('Churn Prediction — Modelo Random Forest | AUC = {:.3f}'.format(auc), fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('images/06_churn_prediction.png', dpi=150, bbox_inches='tight')
plt.show()

## Secao 8 — Simulador de Impacto: Quanto Vale Reter?

Traducao dos resultados do modelo em **valor financeiro concreto** para o negocio.

In [ ]:
# Adicionar probabilidade ao dataset completo
df['prob_churn_modelo'] = rf.predict_proba(df_model.drop('churn',axis=1))[:, 1]
df['risco_churn'] = pd.cut(df['prob_churn_modelo'],
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Baixo', 'Medio', 'Alto'])

# Receita media por segmento (estimativa)
receita_seg = {'Bronze': 85, 'Prata': 180, 'Ouro': 420, 'Diamante': 1200}  # R$/mes

alto_risco = df[df['risco_churn'] == 'Alto']

print('SIMULADOR DE IMPACTO FINANCEIRO')
print('=' * 55)
print(f'Clientes em alto risco de churn: {len(alto_risco):,}')
print()

total_receita_risco = 0
for seg in ordem_seg:
    n_risco = len(alto_risco[alto_risco['segmento'] == seg])
    rec_mensal = receita_seg[seg]
    rec_anual  = n_risco * rec_mensal * 12
    total_receita_risco += rec_anual
    print(f'  {seg:<12}: {n_risco:>5,} clientes | R$ {rec_mensal}/mes | R$ {rec_anual:>12,.0f}/ano em risco')

print(f'\n  TOTAL EM RISCO: R$ {total_receita_risco:,.0f}/ano')
print()

taxa_retencao = 0.25  # campanha retendo 25% dos clientes em risco
receita_preservada = total_receita_risco * taxa_retencao
custo_campanha = len(alto_risco) * 45  # R$45 por cliente abordado
roi = (receita_preservada - custo_campanha) / custo_campanha * 100

print(f'CENARIO DE INTERVENCAO (taxa de retencao: {taxa_retencao:.0%})')
print(f'  Receita preservada:  R$ {receita_preservada:,.0f}/ano')
print(f'  Custo da campanha:   R$ {custo_campanha:,.0f}')
print(f'  ROI da acao:         {roi:.0f}%')

## Secao 9 — Dashboard Executivo Final

In [ ]:
fig = plt.figure(figsize=(22, 16))
fig.patch.set_facecolor('#f8f9fa')
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.38)

# KPI cards
ax_kpi = fig.add_subplot(gs[0, :])
ax_kpi.axis('off')

taxa_churn = df['churn'].mean()
paradoxo_pct = df['paradoxo_nps'].mean()
alto_risco_n = len(df[df['risco_churn']=='Alto'])

kpis = [
    ('Base de Clientes',      f'{N:,}',                              PROTEGE_BLUE),
    ('NPS Geral',             f'{nps_geral:.1f}',                    PROTEGE_GREEN if nps_geral > 0 else PROTEGE_RED),
    ('Taxa de Churn',         f'{taxa_churn:.1%}',                   PROTEGE_RED),
    ('Paradoxo Silencioso',   f'{paradoxo_pct:.1%} dos clientes',    PROTEGE_GOLD),
    ('Em Alto Risco',         f'{alto_risco_n:,} clientes',          PROTEGE_RED),
    ('AUC-ROC Modelo',        f'{auc:.3f}',                          PROTEGE_BLUE),
]

for i, (titulo, valor, cor) in enumerate(kpis):
    x = 0.083 + i * 0.167
    ax_kpi.add_patch(mpatches.FancyBboxPatch((x-0.075, 0.05), 0.148, 0.88,
        boxstyle='round,pad=0.02', facecolor=cor, alpha=0.15, edgecolor=cor, linewidth=2,
        transform=ax_kpi.transAxes))
    ax_kpi.text(x, 0.65, titulo, ha='center', va='center', fontsize=9, color='#555', transform=ax_kpi.transAxes)
    ax_kpi.text(x, 0.28, valor, ha='center', va='center', fontsize=12, fontweight='bold', color=cor, transform=ax_kpi.transAxes)

# NPS por segmento
ax1 = fig.add_subplot(gs[1, 0])
cores_nps2 = [PROTEGE_GREEN if v >= 0 else PROTEGE_RED for v in nps_seg]
bars = ax1.bar(nps_seg.index, nps_seg.values, color=cores_nps2, edgecolor='white')
ax1.bar_label(bars, labels=[f'{v:.1f}' for v in nps_seg.values], padding=3, fontsize=9)
ax1.axhline(0, color='black', linewidth=0.8)
ax1.set_title('NPS por Segmento', fontweight='bold')
ax1.tick_params(axis='x', rotation=15, labelsize=9)

# Satisfacao por produto
ax2 = fig.add_subplot(gs[1, 1])
bars2 = ax2.barh(labels_mf, medias_prod, color=cores_prod, edgecolor='white')
ax2.bar_label(bars2, labels=[f'{v:.2f}' for v in medias_prod], padding=3, fontsize=9)
ax2.axvline(3.5, color='red', linestyle='--', alpha=0.7)
ax2.set_xlim(0, 5.5)
ax2.set_title('Satisfacao por Produto', fontweight='bold')

# Triangulacao
ax3 = fig.add_subplot(gs[1, 2])
tri_vals = df['triangulacao'].value_counts()
ax3.pie(tri_vals.values, labels=tri_vals.index,
        colors=[cores_tri.get(k, PROTEGE_GRAY) for k in tri_vals.index],
        autopct='%1.1f%%', startangle=90,
        wedgeprops={'edgecolor':'white','linewidth':1.5})
ax3.set_title('Triangulacao NPS vs Churn', fontweight='bold')

# Market Fit heatmap
ax4 = fig.add_subplot(gs[1, 3])
sns.heatmap(pivot_mf, annot=True, fmt='.0f', cmap='RdYlGn',
            ax=ax4, vmin=50, vmax=90, linewidths=0.5,
            annot_kws={'fontsize':9}, cbar=False)
ax4.set_title('Score Market Fit', fontweight='bold')
ax4.tick_params(axis='x', rotation=20, labelsize=8)
ax4.tick_params(axis='y', labelsize=8)

# ROC
ax5 = fig.add_subplot(gs[2, 0])
ax5.plot(fpr, tpr, color=PROTEGE_BLUE, linewidth=2)
ax5.plot([0,1],[0,1],'k--', alpha=0.4)
ax5.fill_between(fpr, tpr, alpha=0.1, color=PROTEGE_BLUE)
ax5.set_title(f'ROC Curve (AUC={auc:.3f})', fontweight='bold')
ax5.set_xlabel('FPR'); ax5.set_ylabel('TPR')

# Feature importance top 6
ax6 = fig.add_subplot(gs[2, 1])
imp6 = importancias.tail(6)
ax6.barh(imp6.index, imp6.values*100, color=PROTEGE_BLUE, edgecolor='white', alpha=0.85)
ax6.bar_label(ax6.containers[0], labels=[f'{v*100:.1f}%' for v in imp6.values], padding=3, fontsize=9)
ax6.set_title('Top Drivers de Churn', fontweight='bold')

# Risco por segmento
ax7 = fig.add_subplot(gs[2, 2])
risco_seg = df.groupby(['segmento','risco_churn']).size().unstack(fill_value=0)
risco_seg = risco_seg.reindex(ordem_seg)
risco_seg.plot(kind='bar', ax=ax7, color=[PROTEGE_GREEN,PROTEGE_GOLD,PROTEGE_RED], edgecolor='white')
ax7.set_title('Risco de Churn por Segmento', fontweight='bold')
ax7.tick_params(axis='x', rotation=15, labelsize=9)
ax7.legend(title='Risco', fontsize=8)

# Impacto financeiro
ax8 = fig.add_subplot(gs[2, 3])
impacto = {seg: len(alto_risco[alto_risco['segmento']==seg]) * receita_seg[seg] * 12 / 1e6 for seg in ordem_seg}
bars8 = ax8.bar(list(impacto.keys()), list(impacto.values()),
                color=cores_mf, edgecolor='white')
ax8.bar_label(bars8, labels=[f'R${v:.1f}M' for v in impacto.values()], padding=3, fontsize=9)
ax8.set_title('Receita em Risco por Segmento (R$ Mi)', fontweight='bold')
ax8.tick_params(axis='x', rotation=15, labelsize=9)

fig.suptitle('Dashboard Executivo — Inteligencia de Mercado | Protege Bank',
    fontsize=16, fontweight='bold', y=1.01)
plt.savefig('images/07_dashboard_executivo.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## Conclusoes e Recomendacoes Estrategicas

Celula abaixo gera as conclusoes automaticamente com os valores reais do modelo.

In [ ]:
# ── Conclusoes automaticas ───────────────────────────────────────────────────
paradoxo_n   = df['paradoxo_nps'].sum()
paradoxo_pct = df['paradoxo_nps'].mean() * 100
alto_risco_n = len(df[df['risco_churn'] == 'Alto'])
receita_risco_total = sum([
    len(df[(df['risco_churn']=='Alto') & (df['segmento']==seg)]) * receita_seg[seg] * 12
    for seg in ordem_seg
])

seg_maior_risco = df.groupby('segmento')['prob_churn_modelo'].mean().reindex(ordem_seg).idxmax()
prod_menor_sat  = min(zip(labels_mf, medias_prod), key=lambda x: x[1])[0]
regiao_menor_nps = nps_reg.idxmin()

print("=" * 60)
print("CONCLUSOES — INTELIGENCIA DE MERCADO | PROTEGE BANK")
print("=" * 60)
print()
print(f"  NPS Geral:               {nps_geral:.1f}  —  {'Positivo' if nps_geral > 0 else 'Critico'}")
print(f"  Taxa de Churn:           {df['churn'].mean():.1%}  —  {'Atencao' if df['churn'].mean() > 0.15 else 'Controlado'}")
print(f"  Paradoxo Silencioso:     {paradoxo_pct:.1f}% ({paradoxo_n:,} clientes)  —  Risco Oculto")
print(f"  AUC-ROC Modelo:          {auc:.3f}  —  {'Excelente' if auc > 0.85 else 'Bom'}")
print(f"  Clientes em Alto Risco:  {alto_risco_n:,}  —  Intervencao Urgente")
print(f"  Receita em Risco:        R$ {receita_risco_total/1e6:.1f}M  —  Impacto Critico")
print(f"  ROI da Intervencao:      {roi:.0f}%  —  Viavel")
print()
print("=" * 60)
print()
print("RECOMENDACOES ESTRATEGICAS:")
print()
print(f"  1. URGENTE: Segmento {seg_maior_risco} tem maior probabilidade media de churn.")
print(f"     Acionar campanha de retencao proativa imediatamente.")
print()
print(f"  2. PRODUTO: {prod_menor_sat} tem a menor satisfacao da base.")
print(f"     Investigar dores via pesquisa qualitativa (entrevistas em profundidade).")
print()
print(f"  3. REGIAO: {regiao_menor_nps} tem o pior NPS da rede.")
print(f"     Mapear diferenciais regionais: atendimento, oferta, concorrencia local.")
print()
print(f"  4. PARADOXO: {paradoxo_pct:.1f}% dos clientes nao sao Detratores mas vao embora.")
print(f"     NPS isolado nao captura esse risco. Triangular sempre com dados comportamentais.")
print()
print(f"  5. FINANCEIRO: R$ {receita_preservada/1e6:.1f}M preservados com ROI de {roi:.0f}%.")
print(f"     Campanha de intervencao se paga rapidamente.")
print()
print("=" * 60)

## Secao Extra — Animacoes para LinkedIn

Quatro animacoes geradas a partir dos dados do Protege Bank.
Suba como video no LinkedIn — reproduzem automaticamente no feed.

**Dependencia:** `pip install pillow`

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
print('Imports carregados. Pronto para gerar as animacoes!')

### Animacao 1 — Radar Chart: Score de Saude por Segmento

As 4 dimensoes do Score aparecem progressivamente para cada segmento.
Muito visual e unico no feed do LinkedIn.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

CORES_RADAR = {
    'Bronze':   '#CD7F32',
    'Prata':    '#C0C0C0',
    'Ouro':     '#FFB800',
    'Diamante': '#00BFFF',
}
ordem_seg = ['Bronze','Prata','Ouro','Diamante']

# Scores reais do modelo por segmento
scores_radar = {
    'Bronze':   {'Financeiro': 0.0,  'Reputacao': 0.0,  'Competitividade': 0.0,  'Tendencia': 0.0},
    'Prata':    {'Financeiro': 73.4, 'Reputacao': 80.9, 'Competitividade': 28.0, 'Tendencia': 41.3},
    'Ouro':     {'Financeiro': 74.1, 'Reputacao': 48.9, 'Competitividade': 58.0, 'Tendencia': 81.5},
    'Diamante': {'Financeiro': 100.0,'Reputacao': 100.0,'Competitividade': 76.0, 'Tendencia': 100.0},
}

# Recalcula Bronze a partir dos dados reais
scores_radar['Bronze'] = {
    'Financeiro':     float(df[df['segmento']=='Bronze']['prob_churn_modelo'].apply(lambda x: (1-x)*100).mean()),
    'Reputacao':      float(df[df['segmento']=='Bronze']['nps_score'].apply(lambda x: x/10*100).mean()),
    'Competitividade':float(df[df['segmento']=='Bronze']['n_produtos'].apply(lambda x: x/6*100).mean()),
    'Tendencia':      float(df[df['segmento']=='Bronze']['acessos_app_mes'].apply(lambda x: x/60*100).mean()),
}
scores_radar['Prata'] = {
    'Financeiro':     float(df[df['segmento']=='Prata']['prob_churn_modelo'].apply(lambda x: (1-x)*100).mean()),
    'Reputacao':      float(df[df['segmento']=='Prata']['nps_score'].apply(lambda x: x/10*100).mean()),
    'Competitividade':float(df[df['segmento']=='Prata']['n_produtos'].apply(lambda x: x/6*100).mean()),
    'Tendencia':      float(df[df['segmento']=='Prata']['acessos_app_mes'].apply(lambda x: x/60*100).mean()),
}
scores_radar['Ouro'] = {
    'Financeiro':     float(df[df['segmento']=='Ouro']['prob_churn_modelo'].apply(lambda x: (1-x)*100).mean()),
    'Reputacao':      float(df[df['segmento']=='Ouro']['nps_score'].apply(lambda x: x/10*100).mean()),
    'Competitividade':float(df[df['segmento']=='Ouro']['n_produtos'].apply(lambda x: x/6*100).mean()),
    'Tendencia':      float(df[df['segmento']=='Ouro']['acessos_app_mes'].apply(lambda x: x/60*100).mean()),
}
scores_radar['Diamante'] = {
    'Financeiro':     float(df[df['segmento']=='Diamante']['prob_churn_modelo'].apply(lambda x: (1-x)*100).mean()),
    'Reputacao':      float(df[df['segmento']=='Diamante']['nps_score'].apply(lambda x: x/10*100).mean()),
    'Competitividade':float(df[df['segmento']=='Diamante']['n_produtos'].apply(lambda x: x/6*100).mean()),
    'Tendencia':      float(df[df['segmento']=='Diamante']['acessos_app_mes'].apply(lambda x: x/60*100).mean()),
}

dimensoes = ['Financeiro','Reputacao','Competitividade','Tendencia']
N_DIM = len(dimensoes)
angulos = np.linspace(0, 2*np.pi, N_DIM, endpoint=False).tolist()
angulos += angulos[:1]  # fechar o poligono

fig1, ax1 = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
fig1.patch.set_facecolor('#0f1117')
ax1.set_facecolor('#0f1117')

# Estilo do radar
ax1.set_xticks(angulos[:-1])
ax1.set_xticklabels(dimensoes, color='#aaa', fontsize=11, fontweight='500')
ax1.set_yticks([25, 50, 75, 100])
ax1.set_yticklabels(['25','50','75','100'], color='#555', fontsize=8)
ax1.set_ylim(0, 110)
ax1.grid(color='#1e1e2e', linewidth=0.8)
ax1.spines['polar'].set_color('#333')
ax1.set_title(
    'Protege Bank — Score de Saude por Segmento\nFinanceiro · Reputacao · Competitividade · Tendencia',
    color='white', fontsize=12, fontweight='bold', pad=20
)

# Criar linhas e preenchimentos para cada segmento
linhas_radar = {}
fills_radar  = {}
for seg in ordem_seg:
    cor = CORES_RADAR[seg]
    line, = ax1.plot([], [], color=cor, linewidth=2.5,
                     label=seg, alpha=0.9)
    fill  = ax1.fill([], [], color=cor, alpha=0)
    linhas_radar[seg] = line
    fills_radar[seg]  = fill[0]

ax1.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15),
           fontsize=10, facecolor='#1a1d27', edgecolor='#333',
           labelcolor='white', framealpha=0.9)

N_FRAMES_RADAR = 80
DELAY_POR_SEG  = 15  # frames de delay entre cada segmento

def update_radar(frame):
    for s_idx, seg in enumerate(ordem_seg):
        inicio = s_idx * DELAY_POR_SEG
        progresso = max(0, min((frame - inicio) / N_FRAMES_RADAR, 1.0))
        ease = 1 - (1 - progresso) ** 2

        vals = [scores_radar[seg][d] * ease for d in dimensoes]
        vals += vals[:1]

        linhas_radar[seg].set_data(angulos, vals)
        fills_radar[seg].set_xy(
            np.column_stack([angulos, vals])
        )
        fills_radar[seg].set_alpha(min(ease * 0.25, 0.25))

    return list(linhas_radar.values()) + list(fills_radar.values())

total_frames = N_FRAMES_RADAR + DELAY_POR_SEG * len(ordem_seg) + 30

ani1 = animation.FuncAnimation(
    fig1, update_radar,
    frames=total_frames,
    interval=40,
    blit=False
)

ani1.save('images/anim1_radar_score.gif',
    writer='pillow', fps=25, dpi=120,
    savefig_kwargs={'facecolor': '#0f1117'})

print('Animacao 1 salva: images/anim1_radar_score.gif')
plt.close()

### Animacao 2 — Score de Saude revelado

Barras crescendo progressivamente, revelando o score de cada segmento.

In [ ]:
scores = {'Bronze': 0, 'Prata': 0, 'Ouro': 0, 'Diamante': 0}
scores_finais = {
    'Bronze': df.groupby('segmento')['prob_churn_modelo'].apply(lambda x: (1-x.mean())*100).get('Bronze', 60),
    'Prata':  df.groupby('segmento')['prob_churn_modelo'].apply(lambda x: (1-x.mean())*100).get('Prata', 70),
    'Ouro':   df.groupby('segmento')['prob_churn_modelo'].apply(lambda x: (1-x.mean())*100).get('Ouro', 80),
    'Diamante': df.groupby('segmento')['prob_churn_modelo'].apply(lambda x: (1-x.mean())*100).get('Diamante', 90),
}

CORES_SCORE = ['#CD7F32','#C0C0C0','#FFB800','#00BFFF']
N_FRAMES2 = 60

fig2, ax2 = plt.subplots(figsize=(10, 6))
fig2.patch.set_facecolor('#0f1117')
ax2.set_facecolor('#0f1117')

bars2 = ax2.bar(list(scores_finais.keys()), [0,0,0,0],
                color=CORES_SCORE, edgecolor='none', width=0.55)

ax2.set_ylim(0, 105)
ax2.set_xlim(-0.5, 3.5)
ax2.set_ylabel('Score de Retencao (0-100)', color='#aaa', fontsize=10)
ax2.tick_params(colors='#aaa', labelsize=10)
ax2.spines['bottom'].set_color('#333')
ax2.spines['left'].set_color('#333')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.set_title(
    'Protege Bank — Score de Retencao por Segmento',
    color='white', fontsize=13, fontweight='bold', pad=16
)
ax2.grid(axis='y', color='#1e1e2e', linewidth=0.5)

labels2 = [ax2.text(i, 2, '', ha='center', va='bottom',
                    color='white', fontsize=12, fontweight='bold')
           for i in range(4)]

linha_critica = ax2.axhline(y=75, color='#ff5252', linestyle='--',
                             linewidth=1, alpha=0, label='Meta: 75')
leg2 = ax2.legend(loc='upper left', fontsize=9, facecolor='#1a1d27',
                  edgecolor='#333', labelcolor='white')

def update2(frame):
    progress = min(frame / (N_FRAMES2 * 0.7), 1.0)
    ease = 1 - (1 - progress) ** 3
    for i, (seg, bar, lbl) in enumerate(zip(
            list(scores_finais.keys()), bars2, labels2)):
        val = scores_finais[seg] * ease
        bar.set_height(val)
        if val > 5:
            lbl.set_text(f'{val:.0f}')
            lbl.set_y(val + 1)
    if frame > N_FRAMES2 * 0.6:
        linha_critica.set_alpha(min((frame - N_FRAMES2*0.6) / 10, 0.8))
    return bars2

ani2 = animation.FuncAnimation(fig2, update2,
    frames=N_FRAMES2 + 20, interval=50, blit=False)

ani2.save('images/anim2_score_retencao.gif',
    writer='pillow', fps=20, dpi=120,
    savefig_kwargs={'facecolor': '#0f1117'})

print('Animacao 2 salva: images/anim2_score_retencao.gif')
plt.close()

### Animacao 3 — O Paradoxo Silencioso revelado

Primeiro aparece o NPS — tudo parece ok. Depois os churners aparecem em vermelho sobre os Promotores e Neutros.

In [ ]:
df_anim3 = df.sample(min(2000, len(df)), random_state=42).copy()

fig3, ax3 = plt.subplots(figsize=(11, 7))
fig3.patch.set_facecolor('#0f1117')
ax3.set_facecolor('#0f1117')

# Scatter base — todos os clientes por NPS
nao_churn = df_anim3[df_anim3['churn'] == 0]
churn     = df_anim3[(df_anim3['churn'] == 1) & (df_anim3['nps_categoria'] != 'Detrator')]

sc_base = ax3.scatter(
    nao_churn['nps_score'], nao_churn['acessos_app_mes'],
    c='#378ADD', s=12, alpha=0.4, edgecolors='none', label='Clientes retidos'
)
sc_churn = ax3.scatter(
    churn['nps_score'], churn['acessos_app_mes'],
    c='#ff5252', s=22, alpha=0, edgecolors='none', label='Paradoxo Silencioso'
)

ax3.set_xlabel('NPS Score', color='#aaa', fontsize=10)
ax3.set_ylabel('Acessos ao App / mes', color='#aaa', fontsize=10)
ax3.tick_params(colors='#666', labelsize=8)
ax3.spines['bottom'].set_color('#333')
ax3.spines['left'].set_color('#333')
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)
ax3.grid(color='#1e1e2e', linewidth=0.5)

titulo3 = ax3.set_title(
    'Protege Bank — NPS: tudo parece bem...',
    color='white', fontsize=13, fontweight='bold', pad=14
)
leg3 = ax3.legend(loc='upper right', fontsize=9, facecolor='#1a1d27',
                  edgecolor='#333', labelcolor='white', markerscale=1.5)

txt3 = ax3.text(0.5, 0.08, '', transform=ax3.transAxes,
                ha='center', color='#ff5252', fontsize=13,
                fontweight='bold', alpha=0)

N3 = 80

def update3(frame):
    if frame < 30:
        pass
    elif frame < 30 + N3:
        p = (frame - 30) / N3
        sc_churn.set_alpha(p * 0.85)
        titulo3.set_text('Protege Bank — mas o modelo ve outra historia...')
        txt3.set_alpha(p)
        txt3.set_text(f'{len(churn):,} clientes nao-Detratores vao embora em silencio')
    return sc_churn, txt3

ani3 = animation.FuncAnimation(fig3, update3,
    frames=30 + N3 + 30, interval=50, blit=False)

ani3.save('images/anim3_paradoxo_silencioso.gif',
    writer='pillow', fps=20, dpi=120,
    savefig_kwargs={'facecolor': '#0f1117'})

print('Animacao 3 salva: images/anim3_paradoxo_silencioso.gif')
plt.close()

### Animacao 4 — Curva ROC sendo desenhada

A curva ROC aparece progressivamente revelando o AUC de 0,731.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# Recalcula ROC com os dados do modelo ja treinado
fpr_v, tpr_v, _ = roc_curve(y_test, y_proba)
auc_v = roc_auc_score(y_test, y_proba)

# Suaviza a curva para animacao mais fluida
from scipy.ndimage import uniform_filter1d
fpr_s = uniform_filter1d(fpr_v, size=5)
tpr_s = uniform_filter1d(tpr_v, size=5)

N4 = len(fpr_s)
STEP = max(1, N4 // 120)

fig4, ax4 = plt.subplots(figsize=(8, 7))
fig4.patch.set_facecolor('#0f1117')
ax4.set_facecolor('#0f1117')

ax4.plot([0,1],[0,1],'--', color='#555', linewidth=1, label='Aleatorio (AUC=0.5)')
ax4.set_xlim(-0.02, 1.02)
ax4.set_ylim(-0.02, 1.08)
ax4.set_xlabel('Taxa de Falso Positivo', color='#aaa', fontsize=10)
ax4.set_ylabel('Taxa de Verdadeiro Positivo', color='#aaa', fontsize=10)
ax4.tick_params(colors='#666', labelsize=8)
ax4.spines['bottom'].set_color('#333')
ax4.spines['left'].set_color('#333')
ax4.spines['top'].set_visible(False)
ax4.spines['right'].set_visible(False)
ax4.grid(color='#1e1e2e', linewidth=0.5)
ax4.set_title(
    'Protege Bank — Curva ROC · Modelo de Churn',
    color='white', fontsize=13, fontweight='bold', pad=14
)

line4, = ax4.plot([], [], color='#00BFFF', linewidth=2.5,
                  label=f'Modelo (AUC={auc_v:.3f})')
fill4 = ax4.fill_between([], [], alpha=0)
txt_auc = ax4.text(0.62, 0.12, '', color='#00BFFF', fontsize=15,
                   fontweight='bold', transform=ax4.transAxes, alpha=0)
leg4 = ax4.legend(loc='lower right', fontsize=9, facecolor='#1a1d27',
                  edgecolor='#333', labelcolor='white')

frames4 = list(range(0, N4, STEP)) + [N4-1] * 20

def update4(i):
    global fill4
    idx = frames4[i]
    line4.set_data(fpr_s[:idx], tpr_s[:idx])
    fill4.remove()
    fill4 = ax4.fill_between(fpr_s[:idx], tpr_s[:idx],
                             alpha=0.12, color='#00BFFF')
    progress = idx / (N4 - 1)
    if progress > 0.9:
        txt_auc.set_alpha(min((progress - 0.9) * 10, 1))
        txt_auc.set_text(f'AUC = {auc_v:.3f}')
    return line4, txt_auc

ani4 = animation.FuncAnimation(fig4, update4,
    frames=len(frames4), interval=40, blit=False)

ani4.save('images/anim4_roc_curve.gif',
    writer='pillow', fps=25, dpi=120,
    savefig_kwargs={'facecolor': '#0f1117'})

print('Animacao 4 salva: images/anim4_roc_curve.gif')
plt.close()
print('\nTodas as 4 animacoes geradas com sucesso!')
print('Suba cada GIF como video no LinkedIn.')